<h2>Take Home Test from The Economist Group </h2>

<h3>Coding Test 1</h3>

<p>Consumer Price Index (CPI) series for 7 countries - Brazil, Russia, India, China, Japan, US and Germany</p>

<h4>1. Ingest the below data in R/ Python </h4>

In [2]:
import pandas as pd
df = pd.read_csv('7_Country_CPI_Data.csv')
df.head() 

,Country,Series,1993q1,1993q2,1993q3,1993q4,1994q1,1994q2,1994q3,1994q4,...,2010q3,2010q4,2011q1,2011q2,2011q3,2011q4,2012q1,2012q2,2012q3,2012q4
0,Brazil,Consumer price index (av),6.5,13.5,30.7,75.7,207.5,614.2,932.0,996.0,...,3116.5,3173.8,3248.2,3308.9,3349.9,3386.8,3426.6,3470.0,3513.9,3555.9
1,Russia,Consumer price index (av),0.3,0.5,1.0,1.6,2.4,3.0,3.5,5.0,...,163.6,167.1,173.9,176.7,178.6,180.9,186.2,190.3,192.9,195.4
2,India,Consumer price index (av),52.3,53.4,55.3,57.0,57.2,58.9,61.4,62.6,...,178.3,182.7,186.3,NaN,192.0,194.7,197.3,199.6,202.0,204.3
3,China,Consumer price index (av),55.7,58.7,60.0,62.8,68.1,71.5,75.4,79.6,...,115.5,118.5,120.9,121.7,121.4,123.5,126.0,126.4,126.2,128.4
4,Japan,Consumer price index (av),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,99.2,99.8,99.9,100.2,100.5,100.7,100.9,101.1,101.4,101.7


<h4>2. Impute the values which are missing in the below table using a suitable imputation methodology. Specify the resasons for choosing the particular imputation methodology.</h4>

<p>CPI is a timed-based series -> values change gradually, not randomly.So filling based on nearby values makes sense.</p>
<P>Linear interpolation was used because this method preserves the trend and avoids distortion. Forward and backward fill were applied to handle edge cases where interpolation was not possible.</P>

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 82 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Country  7 non-null      object 
 1   Series   7 non-null      object 
 2   1993q1   6 non-null      float64
 3   1993q2   6 non-null      float64
 4   1993q3   5 non-null      float64
 5   1993q4   6 non-null      float64
 6   1994q1   6 non-null      float64
 7   1994q2   6 non-null      float64
 8   1994q3   6 non-null      float64
 9   1994q4   6 non-null      float64
 10  1995q1   7 non-null      float64
 11  1995q2   7 non-null      float64
 12  1995q3   6 non-null      float64
 13  1995q4   7 non-null      float64
 14  1996q1   7 non-null      float64
 15  1996q2   7 non-null      float64
 16  1996q3   7 non-null      float64
 17  1996q4   7 non-null      float64
 18  1997q1   7 non-null      float64
 19  1997q2   7 non-null      float64
 20  1997q3   7 non-null      float64
 21  1997q4   7 non-null 

In [4]:
# Check missing values
print(df.isnull().sum())

Country    0
Series     0
1993q1     1
1993q2     1
1993q3     2
          ..
2011q4     0
2012q1     0
2012q2     0
2012q3     1
2012q4     0
Length: 82, dtype: int64


In [5]:
# Separate non-numeric columns
meta_cols = df[['Country','Series']]
data = df.drop(columns=['Country','Series'])

In [6]:
# Convert to numeric
data = data.apply(pd.to_numeric, errors = 'coerce')

In [7]:
# Apply linear interpolation 
data_imputed = data.interpolate(method = 'linear', axis = 1)

# Handle edge missing values
data_imputed = data_imputed.bfill(axis = 1).ffill(axis = 1)

In [8]:
# Combine back
df_imputed = pd.concat([meta_cols, data_imputed], axis = 1)

In [9]:
# Verify
print(df_imputed.isnull().sum())

Country    0
Series     0
1993q1     0
1993q2     0
1993q3     0
          ..
2011q4     0
2012q1     0
2012q2     0
2012q3     0
2012q4     0
Length: 82, dtype: int64


<h4>3. Rebase the Consumer Price Index series for all 7 countries such that 2000q1 is the base year (2000q1=100)</h4>

<p>Rebasing was performed to standardize CPI values across countries by setting a common reference period (2000q1 = 100). This allows meaningful comparison of price levels and inflation trends across countries.</p>

In [10]:
# Separate Again
meta_cols = df_imputed[['Country', 'Series']]
data = df_imputed.drop(columns=['Country', 'Series'])

# Get the base year (2000q1 column)
base = data['2000q1']

# Rebase
data_rebased = data.div(base, axis=0) * 100

# Combine back
df_rebased = pd.concat([meta_cols, data_rebased], axis=1)

# Check
print(df_rebased[['Country', '2000q1']])


   Country  2000q1
0   Brazil   100.0
1   Russia   100.0
2    India   100.0
3    China   100.0
4    Japan   100.0
5       US   100.0
6  Germany   100.0


In [11]:
print(df_rebased.head(7))

   Country                     Series     1993q1     1993q2     1993q3  \
0   Brazil  Consumer price index (av)   0.405996   0.843223   1.917552   
1   Russia  Consumer price index (av)   0.641026   1.068376   2.136752   
2    India  Consumer price index (av)  56.115880  57.296137  59.334764   
3    China  Consumer price index (av)  58.693361  61.854584  63.224447   
4    Japan  Consumer price index (av)  98.153547  98.153547  98.153547   
5       US  Consumer price index (av)  84.126984  84.714874  85.126396   
6  Germany  Consumer price index (av)  89.370933  90.130152  90.726681   

      1993q4     1994q1     1994q2     1994q3     1994q4  ...      2010q3  \
0   4.728295  12.960650  38.363523  58.213616  62.211118  ...  194.659588   
1   3.418803   5.128205   6.410256   7.478632  10.683761  ...  349.572650   
2  61.158798  61.373391  63.197425  65.879828  67.167382  ...  191.309013   
3  66.174921  71.759747  75.342466  79.452055  83.877766  ...  121.707060   
4  98.153547  98.15354

In [12]:
print(df_rebased[['Country','2000q1']])

   Country  2000q1
0   Brazil   100.0
1   Russia   100.0
2    India   100.0
3    China   100.0
4    Japan   100.0
5       US   100.0
6  Germany   100.0


In [13]:
# Compare before vs after
print("Before rebasing:")
print(df_imputed[['Country', '2000q1']])

print("\nAfter rebasing:")
print(df_rebased[['Country', '2000q1']])

Before rebasing:
   Country  2000q1
0   Brazil  1601.0
1   Russia    46.8
2    India    93.2
3    China    94.9
4    Japan   102.9
5       US   170.1
6  Germany    92.2

After rebasing:
   Country  2000q1
0   Brazil   100.0
1   Russia   100.0
2    India   100.0
3    China   100.0
4    Japan   100.0
5       US   100.0
6  Germany   100.0


In [14]:
# Check one country trend
country_example = df_rebased[df_rebased['Country'] == 'India']
print(country_example.iloc[:, 20:30]) 

      1997q3     1997q4     1998q1     1998q2     1998q3      1998q4  \
2  83.261803  85.193133  88.519313  90.450644  96.137339  100.429185   

      1999q1     1999q2     1999q3      1999q4  
2  96.459227  96.888412  98.819742  100.858369  


<P>You’ll notice:

Values before 2000q1 < 100 <br/>
Values after 2000q1 > 100 (if inflation increased)</P>

<h4>4. Calculate Consumer Price Index (% change period-on-period; av) using the rebased series calculated in question - 3 above</h4>

<P>Quarter-on-quarter percentage change was calculated to measure short-term inflation trends by comparing CPI values between consecutive quarters.<br/>Useful for recent economic trends</P>
<P>Calculating:
</br>
How much CPI changed from one quarter to the next?. 
This is basically short-term inflation.</P>

<p>If:

2000q1 = 100 </br>
2000q2 = 105

Inflation = (105−100) * 100 = 5% </br>
.............................100
	​

</p>

In [17]:
# Separate again
meta_cols = df_rebased[['Country', 'Series']]
data = df_rebased.drop(columns=['Country', 'Series'])

# Calculate % change across quarters
# pct_change(axis=1) → compares each quarter with previous quarter
# axis=1 → moving left to right (time)
# * 100 → converts into percentage
qoq_change = data.pct_change(axis=1) * 100

# Combine back
df_qoq = pd.concat([meta_cols, qoq_change], axis=1)

# View
print(df_qoq.head())

  Country                     Series  1993q1      1993q2      1993q3  \
0  Brazil  Consumer price index (av)     NaN  107.692308  127.407407   
1  Russia  Consumer price index (av)     NaN   66.666667  100.000000   
2   India  Consumer price index (av)     NaN    2.103250    3.558052   
3   China  Consumer price index (av)     NaN    5.385996    2.214651   
4   Japan  Consumer price index (av)     NaN    0.000000    0.000000   

       1993q4      1994q1      1994q2     1994q3     1994q4  ...    2010q3  \
0  146.579805  174.108322  196.000000  51.742104   6.866953  ...  0.893522   
1   60.000000   50.000000   25.000000  16.666667  42.857143  ...  1.425914   
2    3.074141    0.350877    2.972028   4.244482   1.954397  ...  3.602557   
3    4.666667    8.439490    4.992658   5.454545   5.570292  ...  0.522193   
4    0.000000    0.000000    0.000000   0.000000   0.000000  ... -0.401606   

     2010q4    2011q1    2011q2    2011q3    2011q4    2012q1    2012q2  \
0  1.838601  2.344193  

<p>First column (1993q1 or earliest quarter) will be:
NaN because, No previous quarter to compare with.
Totally normal</p>
<p>Positive value → inflation increased 📈 </br>
Negative value → prices dropped 📉</br>
Zero → no change</p>

<h4>5. Calculate the Consumer Price Index (average annual) using the rebased series calculated in question - 3 above</h4>

<P>Annual average CPI was calculated by averaging quarterly values for each year to analyze long-term inflation trends and smooth out short-term fluctuations.</P>
<P>Converting quarterly CPI into yearly average CPI.

Instead of 4 values per year → you’ll have 1 value per year

Simple intuition

If for a country:

Year	.........Quarters </br>
2000	........100, 105, 110, 115

👉 Annual CPI =

(100+105+110+115)/4=107.5</P>
<P>So we need to:

Extract year </br>
Group quarters by year<br/>
Take average</P>

In [18]:
# Separate again
meta_cols = df_rebased[['Country', 'Series']]
data = df_rebased.drop(columns=['Country', 'Series'])

# Transpose (important step) ~ WHY? -> because currently columns = quarters,.... But want to Group By year, Easier to do when time is in rows.
data_t = data.T

# Extract year from index : "1993q1" -> "1993" ~ WHY?, We need a common key (YEAR) to group 4 quarters together.
data_t['Year'] = data_t.index.str[:4]

# Group by year and take average
annual_avg = data_t.groupby('Year').mean()

# Transpose back
annual_avg = annual_avg.T

# Combine back
df_annual = pd.concat([meta_cols, annual_avg], axis=1)

# View
print(df_annual.head())

  Country                     Series       1993       1994       1995  \
0  Brazil  Consumer price index (av)   1.973766  42.937227  71.278888   
1  Russia  Consumer price index (av)   1.816239   7.425214  22.008547   
2   India  Consumer price index (av)  58.476395  64.404506  71.003219   
3   China  Consumer price index (av)  62.486828  77.608008  90.621707   
4   Japan  Consumer price index (av)  98.153547  98.153547  97.910593   

        1996        1997        1998        1999        2000  ...        2003  \
0  82.509369   88.226109   91.044660   95.466896  102.192380  ...  135.832292   
1  32.585470   37.339744   47.649573   88.568376  106.891026  ...  170.993590   
2  77.360515   82.913090   93.884120   98.256438  102.172747  ...  114.806867   
3  98.129610  100.895680  100.105374   98.656481   98.946259  ...  100.079031   
4  98.202138   99.878523  100.534500  100.170068   99.732750  ...   97.764820   

         2004        2005        2006        2007        2008        2009 

<h4>6. Develop a suitable model to forecast quaterly CPI values for the next two years</h4>

<P>Use past CPI data to predict future CPI values (next 8 quarters). ~for each country.</P>
<P>ARIMA </br>
stands for:

AR (AutoRegressive) → uses past values </br>
I (Integrated) → removes trend (makes data stable)</br>
MA (Moving Average) → smooths noise

In simple words:

It learns from history and extends the pattern forward</P>


In [ ]:
# “For each country → clean its CPI data → fit a time series model → predict next 8 quarters → store results”

In [19]:
from statsmodels.tsa.arima.model import ARIMA   # needed for time series forcasting
import pandas as pd  # data handling
import numpy as np  # numerical operations

forecast_dict = {}   # we need a empty structure to store results. Dictionary is ideal for key (country) -> value (forcast)

for i, row in df_rebased.iterrows(): # Loop over rows, each row = one country; each country needs its own model.
    country = row['Country'] 
    
    # Extract numeric values properly
    data = row.drop(['Country', 'Series'])
    data = pd.to_numeric(data, errors='coerce')   # force numeric
    
    # Remove any remaining NaN (just in case)
    data = data.dropna()
    
    # Convert to series
    series = pd.Series(data.values) # ARIMA expects a 1D sequential structure, .values removes index noice and gives clean array.
    
    try:
        # Fit model
        # p=1 -> depends on last value
        # d=1 -> removes trend (make series stable)
        # q=1 -> smooths noise
        
        model = ARIMA(series, order=(1,1,1)) 
        model_fit = model.fit()
        
        # Step 5: Forecast
        forecast = model_fit.forecast(steps=8) # WHY 8? : 2 YEARS = 8 QUARTERS
        
        forecast_dict[country] = forecast
        
    except Exception as e:
        print(f"Error for {country}: {e}") # If one country fails -> others still run; Helps Debugging.

C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'


<P>The code iterates over each country, preprocesses CPI data to ensure it is numeric and complete, and applies an ARIMA model to capture temporal patterns. Forecasts for the next 8 quarters are generated and stored systematically for further integration.</P>

In [20]:
print(df_rebased.dtypes)

Country     object
Series      object
1993q1     float64
1993q2     float64
1993q3     float64
            ...   
2011q4     float64
2012q1     float64
2012q2     float64
2012q3     float64
2012q4     float64
Length: 82, dtype: object


<h4>7. Generate the output in the same format as the input data provided below. </h4>

<P>Last 8 columns = forcasted values</P>

In [21]:
# Get last quarter
cols = df_rebased.columns[2:]  # excluding Country, Series
last_q = cols[-1]  # forcast should continue from the last known quarter. Example: if last = 2012q4, next should start from 2013q1

year = int(last_q[:4]) # first four characters = year (e.g., "2012")
q = int(last_q[-1]) 

future_quarters = []

for i in range(8):
    q += 1
    if q > 4:
        q = 1
        year += 1
    future_quarters.append(f"{year}q{q}")

print(future_quarters)

['2013q1', '2013q2', '2013q3', '2013q4', '2014q1', '2014q2', '2014q3', '2014q4']


In [22]:
# converts your forecast_dict into a dataFrame, .T = transpose (swap rows and columns).
forecast_df = pd.DataFrame(forecast_dict).T 

# assign column names like: 2013q1, 2013q2,......2014q4
forecast_df.columns = future_quarters

# moves the index (country names) into a column; need Country as a column to merge later
forecast_df.reset_index(inplace=True)

# renames the column from index -> Country
forecast_df.rename(columns={'index': 'Country'}, inplace=True)

In [23]:
# joining your original CPI data with the forecasted future data using the Country column.
final_df = pd.merge(df_rebased, forecast_df, on='Country', how='left')  

In [24]:
print(final_df)

   Country                     Series     1993q1     1993q2     1993q3  \
0   Brazil  Consumer price index (av)   0.405996   0.843223   1.917552   
1   Russia  Consumer price index (av)   0.641026   1.068376   2.136752   
2    India  Consumer price index (av)  56.115880  57.296137  59.334764   
3    China  Consumer price index (av)  58.693361  61.854584  63.224447   
4    Japan  Consumer price index (av)  98.153547  98.153547  98.153547   
5       US  Consumer price index (av)  84.126984  84.714874  85.126396   
6  Germany  Consumer price index (av)  89.370933  90.130152  90.726681   

      1993q4     1994q1     1994q2     1994q3     1994q4  ...      2012q3  \
0   4.728295  12.960650  38.363523  58.213616  62.211118  ...  219.481574   
1   3.418803   5.128205   6.410256   7.478632  10.683761  ...  412.179487   
2  61.158798  61.373391  63.197425  65.879828  67.167382  ...  216.738197   
3  66.174921  71.759747  75.342466  79.452055  83.877766  ...  132.982086   
4  98.153547  98.15354

In [25]:
final_df.to_csv("output1_Tanisha_Jain.csv", index=False)